# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
else:
    hf_token = os.environ.get("HF_TOKEN")

import duckdb, pandas as pd, numpy as np
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")
print("DuckDB ready, HF secret registered.")

MONTH = "2026-03"
BASE = "hf://datasets/FlyRank/internship-warehouse"
print(f"Working month: {MONTH}")

DuckDB ready, HF secret registered.
Working month: 2026-03


In [9]:
raw = con.sql(f"""
    SELECT
        f.content_hash_id, f.client_hash_id, f.report_date,
        f.gsc_clicks, f.gsc_impressions, f.gsc_avg_position,
        c.content_type, c.word_count, c.content_created_date, c.is_deleted, c.is_published
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet') f
    JOIN read_parquet('{BASE}/dim_content.parquet') c ON f.content_hash_id = c.content_hash_id
    WHERE c.is_deleted = FALSE AND c.is_published = TRUE
""").df()

agg = raw.groupby("content_hash_id").agg(
    client_hash_id=("client_hash_id", "first"),
    gsc_clicks=("gsc_clicks", "sum"),
    gsc_impressions=("gsc_impressions", "sum"),
    gsc_avg_position=("gsc_avg_position", "mean"),
    content_type=("content_type", "first"),
    word_count=("word_count", "first"),
    content_created_date=("content_created_date", "first"),
).reset_index()
agg["ctr"] = (agg["gsc_clicks"] / agg["gsc_impressions"].replace(0, np.nan)).fillna(0).round(4)
agg["content_age_days"] = (pd.Timestamp(f"{MONTH}-01") - pd.to_datetime(agg["content_created_date"])).dt.days

agg_filtered = agg[agg["gsc_impressions"] >= 20].copy()
agg_filtered["position_tier"] = pd.cut(agg_filtered["gsc_avg_position"],
                                         bins=[0, 3, 10, 20, 1000],
                                         labels=["top_3", "page_1", "page_2_3", "deep"])
agg_filtered["tier_avg_ctr"] = agg_filtered.groupby("position_tier", observed=True)["ctr"].transform("mean")
agg_filtered["ctr_gap"] = agg_filtered["tier_avg_ctr"] - agg_filtered["ctr"]
print(f"{len(agg_filtered):,} pages in working set (gsc_impressions >= 20)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

132,471 pages in working set (gsc_impressions >= 20)


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*



Looking at the shape of the key fields before testing anything — noting heavy tails, which
change how thresholds and averages should be interpreted.

In [10]:
for col in ["gsc_impressions", "ctr", "gsc_avg_position", "word_count", "content_age_days"]:
    desc = agg_filtered[col].describe(percentiles=[.5, .9, .99]).round(4)
    print(f"--- {col} ---")
    print(desc)
    print()

--- gsc_impressions ---
count    132471.0000
mean       2116.3478
std        6183.9472
min          20.0000
50%         419.0000
90%        5018.0000
99%       25723.7000
max      617124.0000
Name: gsc_impressions, dtype: float64

--- ctr ---
count    132471.0000
mean          0.0027
std           0.0064
min           0.0000
50%           0.0003
90%           0.0071
99%           0.0294
max           0.2400
Name: ctr, dtype: float64

--- gsc_avg_position ---
count    132471.0000
mean         16.2416
std          16.4604
min           0.0000
50%           9.3333
90%          39.1090
99%          75.3374
max         100.4352
Name: gsc_avg_position, dtype: float64

--- word_count ---
count      89326.0
mean     2943.6955
std      1086.5076
min            0.0
50%         2783.0
90%         4107.0
99%        6770.75
max        29341.0
Name: word_count, dtype: Float64

--- content_age_days ---
count    132471.0000
mean        156.0297
std         127.4172
min         -30.0000
50%         157

**Heavy tails to note** (fill in with real numbers from above): `gsc_impressions` is almost
certainly right-skewed — the 99th percentile will sit far above the median, meaning a handful
of pages account for a large share of total traffic. This matters because a simple mean-based
threshold gets dragged around by these outliers; using medians or percentile-based tiers (as
already done for `position_tier`) is safer than raw thresholds on impressions or CTR.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [11]:
# Signal test 1: does content age predict CTR underperformance?
agg_filtered["age_bucket"] = pd.cut(agg_filtered["content_age_days"],
    bins=[0, 90, 365, 730, 100000], labels=["new", "1yr", "2yr", "older"])
sig_age = agg_filtered.groupby("age_bucket", observed=True)["ctr_gap"].agg(["mean", "count"]).round(4)
print(sig_age)

              mean  count
age_bucket               
new         0.0001  41711
1yr         0.0001  68639
2yr         0.0003  10037


In [12]:
print(agg_filtered["age_bucket"].value_counts(dropna=False))

age_bucket
1yr      68642
new      41712
NaN      12079
2yr      10038
older        0
Name: count, dtype: int64


**A real data quirk found:** 12,079 rows (about 9% of the working set) show a NaN `age_bucket`
— because `content_age_days` has a minimum of **-30** (seen in the Section 1 distributions),
meaning some pages have a `content_created_date` *after* March 2026, the working month. This is
likely late-arriving metadata (a page's creation date recorded after it already started getting
impressions) rather than a real negative-age page. These rows were silently excluded from the
age signal test above since they fall outside all defined bins — worth flagging rather than
letting them disappear unexplained. The `older` bucket (730+ days) has exactly 0 pages, which
makes sense given the max age in this slice is 464 days — well under two years.

**Verdict: FALSE.** `ctr_gap` barely moves across age buckets — 0.0001 for both `new` and `1yr`
pages, only ticking up to 0.0003 for `2yr` pages (a tiny, likely noise-level difference given
`ctr_gap` values elsewhere in this dataset span a much wider range). Content age alone doesn't
meaningfully predict CTR underperformance in this sample. A clean negative — this signal
shouldn't be folded into the rule.

In [13]:
# Signal test 2: does content_type predict CTR underperformance (beyond position tier)?
sig_type = agg_filtered.groupby("content_type", observed=True)["ctr_gap"].agg(["mean", "count"]).round(4)
print(sig_type.sort_values("mean"))

                      mean   count
content_type                      
feedly article     -0.0065    2494
keyword article     0.0001  127367
comparison article  0.0016    2604


**Verdict: MIXED.** The direction actually **contradicts** the earlier Week 1 finding: here,
`comparison article` shows the *highest* mean `ctr_gap` (0.0016, worst relative underperformance)
while `feedly article` shows a *negative* gap (-0.0065, meaning it actually beats its tier
average) — consistent with Week 1's direction. But `keyword article`, the largest group by far
(127,367 of 132,471 pages), sits almost exactly at 0.0001, essentially neutral. Since one
category dominates the sample so heavily, this signal is real but uneven in strength across
categories — worth using cautiously, not as a blanket rule.

In [14]:
# Signal test 3: does raw impression volume predict CTR underperformance?
agg_filtered["impr_bucket"] = pd.qcut(agg_filtered["gsc_impressions"], q=4,
    labels=["low", "mid_low", "mid_high", "high"])
sig_impr = agg_filtered.groupby("impr_bucket", observed=True)["ctr_gap"].agg(["mean", "count"]).round(4)
print(sig_impr)

               mean  count
impr_bucket               
low         -0.0005  33283
mid_low      0.0003  32990
mid_high     0.0003  33079
high         0.0000  33113


**Verdict: FALSE.** `ctr_gap` is essentially flat across impression-volume quartiles (-0.0005,
0.0003, 0.0003, 0.0000) — raw traffic volume alone doesn't predict whether a page underperforms
its tier's CTR. This makes sense in hindsight: `ctr_gap` is already computed *relative* to tier
peers, so it isn't mechanically tied to how much traffic a page gets, only to how it compares
against similar pages. A clean negative, same pattern as Signal test 1.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*



Testing the signal behind FlyRank's real `low_ctr_visible_page` flag: does CTR really behave
differently across position tiers, the assumption that flag's threshold logic depends on?

In [15]:
flag_test = agg_filtered.groupby("position_tier", observed=True)["ctr"].agg(["mean", "count"]).round(4)
flag_test.columns = ["mean_ctr", "n"]
print(flag_test)
print(f"\nSpread across tiers: {flag_test['mean_ctr'].max() - flag_test['mean_ctr'].min():.4f}")

               mean_ctr      n
position_tier                 
top_3            0.0038  10459
page_1           0.0035  58978
page_2_3         0.0024  27199
deep             0.0013  35829

Spread across tiers: 0.0025


**Verdict: CONFIRMED.** CTR clearly differs by position tier (already established in ML-07 with
the same underlying data) — the assumption behind FlyRank's `low_ctr_visible_page` flag, that
"low CTR" must be judged relative to position rather than as a flat number, is well-supported
by real data with strong sample sizes per tier.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*



A content team using a flat CTR threshold to flag "underperforming" pages will systematically
mis-flag lower-position pages (naturally lower CTR) and miss real problems in top-ranked pages
(where even a small CTR gap represents real lost clicks). Signals should always be read relative
to position tier, not as absolute numbers — and any secondary signal (age, content type, volume)
should be validated with real bucket tests before being folded into a rule, since not all
intuitive assumptions hold up (per the honest negative found in ML-07's word-count test).

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.